In [ ]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver
from rich import print


#1. 状態を宣言
class OverAllState(TypedDict):
    username: str


#2. ノードを宣言
def node_a(state: OverAllState) -> OverAllState:
    username = interrupt("お名前を入力してください")
    return {
        "username": username
    }


#3. グラフを構築
builder = StateGraph(state_schema=OverAllState)

builder.add_node("node_a", node_a)

builder.add_edge(START, "node_a")
builder.add_edge("node_a", END)

#4. 中断機能を使うには => チェックポインターの設定が必須
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

from IPython.display import display

display(graph)

config = {"configurable": {"thread_id": "123"}}
interrupt_res = graph.invoke({}, config=config)
print(interrupt_res)





In [ ]:
# 6. 実行を再開
resume_map = {}
for i in interrupt_res['__interrupt__']:
    user_input = input(f"{i.value}:")
    resume_map[i.id] = user_input

resumed_res = graph.invoke(Command(resume=resume_map), config=config)
print(resumed_res)
